In [8]:
!pip install optuna fair-esm -q
!wget https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz -q
!tar -xzf cd-hit-v4.8.1-2019-0228.tar.gz
!cd cd-hit-v4.8.1-2019-0228 && make -j4 > /dev/null
!./cd-hit-v4.8.1-2019-0228/cd-hit -h | head -3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.1 MB/s eta 0:00:00
		====== CD-HIT version 4.8.1 (built on Jul  9 2026) ======

Usage: ./cd-hit-v4.8.1-2019-0228/cd-hit [Options] 


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, f1_score
import hashlib, re, os, pickle, zipfile

optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTPUT_DIR = "AMP_model1_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv("/kaggle/input/datasets/amirtesh/amp-types-classification/AMP_features_filtered.csv")
before = len(df)
df = df[df["modlamp_Length"] >= 11].reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} (11aa floor)")

df["seq_id"] = df["sequence"].apply(lambda s: f"seq_{hashlib.md5(s.encode()).hexdigest()[:12]}")
assert df["seq_id"].duplicated().sum() == 0, "Hash collision or duplicate sequence - investigate before continuing"
print(f"Stable seq_id assigned to {len(df)} sequences, 0 collisions confirmed")

Rows: 17323 -> 14817 (11aa floor)
Stable seq_id assigned to 14817 sequences, 0 collisions confirmed


In [4]:
fasta_path = f"{OUTPUT_DIR}/AMP_filtered.fasta"
df_sorted = df.sort_values("modlamp_Length", ascending=False)

with open(fasta_path, "w") as f:
    for _, row in df_sorted.iterrows():
        f.write(f">{row['seq_id']}\n{row['sequence']}\n")

print(f"Wrote {len(df_sorted)} sequences, sorted by length descending")
!head -4 {fasta_path}

Wrote 14817 sequences, sorted by length descending
>seq_a125cb9f4d7a
MPRLKVKLVKSPIGYPKDQKAALKALGLRRLQQERVLEDTPAIRGNVEKVAHLVRVEVVE
>seq_3c753d866cae
RPEIKKKNVFSKPGYCPEYRVPCPFVLIPKCRRDKGCKDALKCCFFYCQMRCVDPWESPE


In [5]:
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

!./cd-hit-v4.8.1-2019-0228/cd-hit -i {fasta_path} -o {OUTPUT_DIR}/AMP_clustered -c 0.5 -n 3 -M 4000 -T 1 -d 0 \
    2>&1 | tee {OUTPUT_DIR}/cdhit_log.txt
!grep -i "shortest\|finished\|clusters" {OUTPUT_DIR}/cdhit_log.txt

hash1 = file_hash(f"{OUTPUT_DIR}/AMP_clustered.clstr")
print(f"Run 1 hash: {hash1}")

!./cd-hit-v4.8.1-2019-0228/cd-hit -i {fasta_path} -o {OUTPUT_DIR}/AMP_clustered_check -c 0.5 -n 3 -M 4000 -T 1 -d 0 > /dev/null 2>&1
hash2 = file_hash(f"{OUTPUT_DIR}/AMP_clustered_check.clstr")
print(f"Run 2 hash: {hash2}")
assert hash1 == hash2, "CD-HIT nondeterministic even with sorted input - stop and investigate further before proceeding"
print("CD-HIT confirmed deterministic on this input.")

/bin/bash: line 1: ./cd-hit-v4.8.1-2019-0228/cd-hit: Permission denied
Run 1 hash: 509864806ff87615ae1126a7dcbbd217
Run 2 hash: 509864806ff87615ae1126a7dcbbd217
CD-HIT confirmed deterministic on this input.


In [6]:
cluster_map = {}
current_cluster = None
with open(f"{OUTPUT_DIR}/AMP_clustered.clstr") as f:
    for line in f:
        if line.startswith(">Cluster"):
            current_cluster = int(line.strip().split()[-1])
        else:
            m = re.search(r">(seq_[a-f0-9]+)\.\.\.", line)
            if m:
                cluster_map[m.group(1)] = current_cluster

df["cluster_id"] = df["seq_id"].map(cluster_map)
assert df["cluster_id"].isna().sum() == 0, "Unassigned sequences - stop and diagnose"
assert len(cluster_map) == len(df), "Count mismatch - stop and diagnose"
print(f"Clusters: {df['cluster_id'].nunique()} from {len(df)} sequences")

unique_clusters = sorted(df["cluster_id"].unique())
rng = np.random.RandomState(42)
rng.shuffle(unique_clusters)

n = len(unique_clusters)
train_c = set(unique_clusters[:int(0.8*n)])
val_c   = set(unique_clusters[int(0.8*n):int(0.9*n)])
test_c  = set(unique_clusters[int(0.9*n):])
df["split"] = df["cluster_id"].apply(lambda c: "train" if c in train_c else ("val" if c in val_c else "test"))

print(df["split"].value_counts())
print(df.groupby("split")[["gram_positive","gram_negative","fungal"]].mean())

df.to_csv(f"{OUTPUT_DIR}/AMP_features_SPLIT_FINAL.csv", index=False)
print(f"\nSaved fixed split to {OUTPUT_DIR}/AMP_features_SPLIT_FINAL.csv")

Clusters: 2077 from 14817 sequences
split
train    11703
val       1645
test      1469
Name: count, dtype: int64
       gram_positive  gram_negative    fungal
split                                        
test        0.726344       0.759020  0.491491
train       0.816799       0.842690  0.362728
val         0.817021       0.830395  0.399392

Saved fixed split to AMP_model1_final/AMP_features_SPLIT_FINAL.csv


In [9]:
import torch, esm
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU - check Kaggle accelerator setting"

model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model = model.to(device).eval()
batch_converter = alphabet.get_batch_converter()

def get_embeddings(sequences, batch_size=32):
    all_emb = []
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = [(str(j), s) for j, s in enumerate(sequences[i:i+batch_size])]
        _, _, tokens = batch_converter(batch)
        tokens = tokens.to(device)
        with torch.no_grad():
            results = model(tokens, repr_layers=[12])
        reps = results["representations"][12]
        for k, (_, s) in enumerate(batch):
            all_emb.append(reps[k, 1:len(s)+1].mean(0).cpu().numpy())
    return all_emb

embeddings = get_embeddings(df["sequence"].tolist())
assert len(embeddings) == len(df), "Count mismatch - stop and debug"

emb_matrix = np.vstack(embeddings)
esm2_cols = [f"esm2_{i}" for i in range(emb_matrix.shape[1])]
emb_df = pd.DataFrame(emb_matrix, columns=esm2_cols, index=df.index)
df = pd.concat([df, emb_df], axis=1)

df.to_csv(f"{OUTPUT_DIR}/AMP_features_SPLIT_FINAL.csv", index=False)  
print(f"df now {df.shape}, saved with ESM2 embeddings included")

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt


100%|██████████| 464/464 [00:26<00:00, 17.83it/s]


df now (14817, 521), saved with ESM2 embeddings included


In [10]:
df = pd.read_csv(f"{OUTPUT_DIR}/AMP_features_SPLIT_FINAL.csv")  

biophysical_cols = [c for c in df.columns if c.startswith("frac_") or c.startswith("modlamp_")
                     or c == "gravy" or c.startswith("ss_")]
esm2_cols = [c for c in df.columns if c.startswith("esm2_")]
combined_cols = biophysical_cols + esm2_cols

labels = ["gram_positive", "gram_negative", "fungal"]

train_df = df[df["split"]=="train"].reset_index(drop=True)
val_df   = df[df["split"]=="val"].reset_index(drop=True)
test_df  = df[df["split"]=="test"].reset_index(drop=True)

print(f"Biophysical: {len(biophysical_cols)} features")
print(f"ESM2: {len(esm2_cols)} features")
print(f"Combined: {len(combined_cols)} features")
print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

Biophysical: 34 features
ESM2: 480 features
Combined: 514 features
Split sizes: train=11703, val=1645, test=1469


In [11]:
def optimize_threshold(y_true, y_prob, metric_fn=matthews_corrcoef):
    best_t, best_s = 0.5, -1
    for t in np.arange(0.05, 0.96, 0.01):
        s = metric_fn(y_true, (y_prob >= t).astype(int))
        if s > best_s: best_s, best_t = s, t
    return best_t, best_s

def make_objective(X_tr, y_tr, X_v, y_v, spw):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "scale_pos_weight": spw,
            "eval_metric": "aucpr", "tree_method": "hist", "device": "cuda",
            "random_state": 42, "n_jobs": -1, "verbosity": 0,
            "early_stopping_rounds": 40,
        }
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)
        return matthews_corrcoef(y_v, m.predict(X_v))
    return objective

def run_feature_set(feature_cols, name, n_trials=60):
    X_train, X_val, X_test = train_df[feature_cols], val_df[feature_cols], test_df[feature_cols]
    models, rows = {}, []

    for label in labels:
        spw = (len(train_df) - train_df[label].sum()) / train_df[label].sum()
        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42, n_startup_trials=15))
        study.optimize(make_objective(X_train, train_df[label], X_val, val_df[label], spw),
                        n_trials=n_trials, show_progress_bar=False)

        bp = study.best_params
        bp.update({"scale_pos_weight": spw, "eval_metric": "aucpr", "tree_method": "hist",
                   "device": "cuda", "random_state": 42, "n_jobs": -1, "verbosity": 0,
                   "early_stopping_rounds": 40})
        model = xgb.XGBClassifier(**bp)
        model.fit(X_train, train_df[label], eval_set=[(X_val, val_df[label])], verbose=False)
        models[label] = model

        p_val, p_test = model.predict_proba(X_val)[:,1], model.predict_proba(X_test)[:,1]
        thresh, _ = optimize_threshold(val_df[label].values, p_val)
        pred_bin = (p_test >= thresh).astype(int)

        rows.append({
            "feature_set": name, "label": label, "threshold": thresh,
            "test_AUROC": roc_auc_score(test_df[label], p_test),
            "test_AUPRC": average_precision_score(test_df[label], p_test),
            "test_MCC": matthews_corrcoef(test_df[label], pred_bin),
            "test_F1": f1_score(test_df[label], pred_bin),
        })
        print(f"  [{name}] {label}: val MCC={study.best_value:.4f}, test AUROC={rows[-1]['test_AUROC']:.4f}, test MCC={rows[-1]['test_MCC']:.4f}")

    return pd.DataFrame(rows), models

print("=== Biophysical only ===")
bio_results, bio_models = run_feature_set(biophysical_cols, "biophysical")
print("\n=== ESM2 only ===")
esm2_results, esm2_models = run_feature_set(esm2_cols, "esm2")
print("\n=== Combined ===")
comb_results, comb_models = run_feature_set(combined_cols, "combined")

final_table = pd.concat([bio_results, esm2_results, comb_results], ignore_index=True)
final_table.to_csv(f"{OUTPUT_DIR}/three_way_ablation_results.csv", index=False)
print("\n=== FULL COMPARISON TABLE (this is the one test-set look) ===")
print(final_table.to_string(index=False))

=== Biophysical only ===
  [biophysical] gram_positive: val MCC=0.5565, test AUROC=0.8102, test MCC=0.4264
  [biophysical] gram_negative: val MCC=0.5407, test AUROC=0.8609, test MCC=0.5602
  [biophysical] fungal: val MCC=0.5156, test AUROC=0.8363, test MCC=0.5145

=== ESM2 only ===
  [esm2] gram_positive: val MCC=0.5362, test AUROC=0.8065, test MCC=0.4490
  [esm2] gram_negative: val MCC=0.5569, test AUROC=0.8825, test MCC=0.6307
  [esm2] fungal: val MCC=0.4398, test AUROC=0.7993, test MCC=0.4369

=== Combined ===
  [combined] gram_positive: val MCC=0.5352, test AUROC=0.8072, test MCC=0.4831
  [combined] gram_negative: val MCC=0.5245, test AUROC=0.8771, test MCC=0.6070
  [combined] fungal: val MCC=0.4522, test AUROC=0.8195, test MCC=0.4835

=== FULL COMPARISON TABLE (this is the one test-set look) ===
feature_set         label  threshold  test_AUROC  test_AUPRC  test_MCC  test_F1
biophysical gram_positive       0.50    0.810228    0.906989  0.426379 0.858300
biophysical gram_negative   

In [12]:
def bootstrap_auroc_ci(y_true, y_prob, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2: continue
        scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(scores, [2.5, 97.5]), np.mean(scores)

for label, model_dict, name in [("gram_negative", bio_models, "biophysical"), ("gram_negative", esm2_models, "esm2"),
                                  ("fungal", bio_models, "biophysical"), ("fungal", esm2_models, "esm2")]:
    cols = biophysical_cols if name == "biophysical" else esm2_cols
    p = model_dict[label].predict_proba(test_df[cols])[:, 1]
    ci, mean = bootstrap_auroc_ci(test_df[label].values, p)
    print(f"{label} ({name}): AUROC {mean:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")

gram_negative (biophysical): AUROC 0.861 [0.839, 0.882]
gram_negative (esm2): AUROC 0.883 [0.861, 0.904]
fungal (biophysical): AUROC 0.836 [0.815, 0.854]
fungal (esm2): AUROC 0.799 [0.776, 0.820]


In [13]:
final_model1 = {
    "gram_positive": {"model": bio_models["gram_positive"], "features": biophysical_cols, "feature_set_name": "biophysical"},
    "gram_negative": {"model": esm2_models["gram_negative"], "features": esm2_cols, "feature_set_name": "esm2"},
    "fungal": {"model": bio_models["fungal"], "features": biophysical_cols, "feature_set_name": "biophysical"},
}

final_summary = final_table[
    ((final_table["feature_set"]=="biophysical") & (final_table["label"]=="gram_positive")) |
    ((final_table["feature_set"]=="esm2") & (final_table["label"]=="gram_negative")) |
    ((final_table["feature_set"]=="biophysical") & (final_table["label"]=="fungal"))
]
print("=== LOCKED MODEL 1 - SELECTED VIA VAL MCC, REPORTED ON TEST (one look) ===")
print(final_summary.to_string(index=False))

=== LOCKED MODEL 1 - SELECTED VIA VAL MCC, REPORTED ON TEST (one look) ===
feature_set         label  threshold  test_AUROC  test_AUPRC  test_MCC  test_F1
biophysical gram_positive       0.50    0.810228    0.906989  0.426379 0.858300
biophysical        fungal       0.48    0.836274    0.836393  0.514466 0.757308
       esm2 gram_negative       0.52    0.882483    0.949302  0.630745 0.909991


In [15]:
import shutil
import os
import pickle
import zipfile

with open(f"{OUTPUT_DIR}/Model1_final.pkl", "wb") as f:
    pickle.dump(final_model1, f)

final_summary.to_csv(f"{OUTPUT_DIR}/Model1_final_summary.csv", index=False)
final_table.to_csv(f"{OUTPUT_DIR}/three_way_ablation_full_results.csv", index=False) 
all_models = {"biophysical": bio_models, "esm2": esm2_models, "combined": comb_models}
with open(f"{OUTPUT_DIR}/all_ablation_models.pkl", "wb") as f:
    pickle.dump(all_models, f)

with open(f"{OUTPUT_DIR}/feature_columns.txt", "w") as f:
    f.write("BIOPHYSICAL:\n" + "\n".join(biophysical_cols) + "\n\n")
    f.write("ESM2:\n" + "\n".join(esm2_cols) + "\n\n")
    f.write("COMBINED:\n" + "\n".join(combined_cols) + "\n")

df.to_csv(f"{OUTPUT_DIR}/AMP_features_SPLIT_FINAL.csv", index=False)

train_df.to_csv(f"{OUTPUT_DIR}/train_split.csv", index=False)
val_df.to_csv(f"{OUTPUT_DIR}/val_split.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test_split.csv", index=False)


print("Contents of output directory before zipping:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" ", f)

zip_path = "AMP_Model1_COMPLETE.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            zf.write(filepath, os.path.relpath(filepath, OUTPUT_DIR))

print(f"\nZipped: {zip_path}, size: {os.path.getsize(zip_path)/1e6:.2f} MB")

Contents of output directory before zipping:
  AMP_clustered
  AMP_clustered.clstr
  AMP_clustered_check
  AMP_clustered_check.clstr
  AMP_features_SPLIT_FINAL.csv
  AMP_filtered.fasta
  Model1_final.pkl
  Model1_final_summary.csv
  all_ablation_models.pkl
  cdhit_log.txt
  feature_columns.txt
  test_split.csv
  three_way_ablation_full_results.csv
  three_way_ablation_results.csv
  train_split.csv
  val_split.csv

Zipped: AMP_Model1_COMPLETE.zip, size: 78.65 MB
